# MNIST 持续学习：EWC 与样本回放

先把十个数字分成五组，每组两个类别，再让同一个网络依次学习。学会下一组数字后，前面学过的数字还能识别吗？

本页比较普通顺序训练、EWC 和样本回放，观察新任务学习与旧任务保持。Task-IL 在测试时提供任务编号，只需区分该任务的两个类别；Class-IL 不提供任务编号，需要从所有已见类别中作出选择。

[本课说明与运行步骤](README.md)


## 1. 导入与随机种子

从本页顶部按顺序执行。随机种子控制任务划分、模型初始化和训练采样；重新比较一组设置时，也需要重新初始化模型。


In [ ]:
from biai.paths import DATA_DIR
from biai.reproducibility import seed_everything, parameter_digest

SEED = 0
seed_everything(SEED, deterministic=True)
USE_NETWORK_DATA = False
DATASET_DIR = DATA_DIR / "network" if USE_NETWORK_DATA else DATA_DIR

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

## 2. 训练设置

本页固定为 5 个任务，每个任务 2 类。`epochs_per_task` 默认是 5：每个任务按训练集大小安排 5 轮更新。普通顺序训练和 EWC 每轮使用全部当前样本；回放对照从第二个任务起，用旧样本替换批次中的部分当前样本，更新次数保持一致。

`ewc_lambda` 控制 EWC 惩罚的强度，`fisher_samples` 控制估计参数重要性时抽取的样本数；`fisher_batch_size=1` 表示逐样本计算梯度平方。回放缓冲区由 `replay_capacity` 控制，默认在所有旧任务间合计保存 500 张图像。


In [ ]:
CONFIG = {
    'data_dir': DATASET_DIR,
    'download_data': USE_NETWORK_DATA,
    'device': device,

    # Task layout.
    'num_tasks': 5,
    'classes_per_task': 2,

    # Model and training settings.
    'hidden_units': 400,
    'batch_size': 128,
    'learning_rate': 0.001,
    'epochs_per_task': 5,
    'fisher_batch_size': 1,

    # Weight of the EWC penalty in the training loss.
    'ewc_lambda': 1000,
    'fisher_samples': 1024,
    'replay_capacity': 500,  # Total stored images across old tasks, not per class.
}

print("\n 配置参数:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. 划分任务

先打乱数字 0–9，再按顺序两两分组。打印出的类别表决定后面的任务顺序。数据集保留原始数字标签，训练函数再把各任务的两个标签映射到相邻的输出位置。


In [ ]:
# Shuffle the digits once, then keep this task order throughout training.
all_classes = list(range(10))
np.random.shuffle(all_classes)
task_classes = [all_classes[i*2:(i+1)*2] for i in range(5)]

# Optional fixed pairing; use this assignment instead of the shuffled one.
# task_classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]

print("\n 任务类别分配:")
for i, classes in enumerate(task_classes):
    print(f"  任务 {i+1}: 类别 {classes}")


## 4. 加载数据

从 MNIST 训练集和测试集中分别筛选每个任务的类别，得到对应的子集。课程包已经包含 MNIST；保持 `USE_NETWORK_DATA = False` 会直接读取包内数据，改为 `True` 后则下载到 `data/network/` 并使用该副本。测试样本不参与参数更新。


In [ ]:
def load_split_mnist(task_classes, data_dir=DATA_DIR, download=False):
    """Load MNIST and split it into tasks without changing digit labels."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = datasets.MNIST(data_dir, train=True, download=download, transform=transform)
    test_dataset = datasets.MNIST(data_dir, train=False, download=download, transform=transform)
    
    train_tasks, test_tasks = [], []
    
    for classes in task_classes:
        # Keep the original digit labels in each subset.
        train_idx = [i for i, label in enumerate(train_dataset.targets) if label in classes]
        test_idx = [i for i, label in enumerate(test_dataset.targets) if label in classes]
        train_tasks.append((Subset(train_dataset, train_idx), classes))
        test_tasks.append((Subset(test_dataset, test_idx), classes))
    
    return train_tasks, test_tasks

print("\n 加载 MNIST 数据集...")
train_tasks, test_tasks = load_split_mnist(
    task_classes, CONFIG['data_dir'], CONFIG['download_data']
)
print(f" 加载完成: {len(train_tasks)} 个任务")


## 5. 查看任务样本

每行展示一个任务中的两个类别，每类取两张图。结合下方的样本数，检查类别分组和数据划分是否符合预期。


In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(12, 15))
fig.suptitle('Sample Images from Each Task', fontsize=16, fontweight='bold')

for task_id, (train_dataset, classes) in enumerate(train_tasks):
    for class_idx, cls in enumerate(classes):
        indices = [i for i, (img, label) in enumerate(train_dataset) if label == cls]
        if indices:
            for sample_idx, idx in enumerate(indices[:2]):
                img, _ = train_dataset[idx]
                ax = axes[task_id, class_idx*2 + sample_idx]
                ax.imshow(img.squeeze().cpu().numpy(), cmap='gray')
                ax.set_title(f'Task {task_id+1}, Class {cls}')
                ax.axis('off')

plt.tight_layout()
plt.show()

print("\n 数据集大小:")
print(f"{'任务':<8} {'类别':<15} {'训练集':<15} {'测试集':<15}")
print("-" * 53)
for task_id, (train_dataset, classes) in enumerate(train_tasks):
    test_dataset, _ = test_tasks[task_id]
    print(
        f"Task {task_id+1:<2} {str(classes):<15} "
        f"{len(train_dataset):<15} {len(test_dataset):<15}"
    )
print("-" * 53)
print(
    f"{'Total':<8} {'':<15} "
    f"{sum(len(train_dataset) for train_dataset, _ in train_tasks):<15} "
    f"{sum(len(test_dataset) for test_dataset, _ in test_tasks):<15}"
)


## 6. 分类网络

图像展平后经过两个各含 400 个单元的 ReLU 隐藏层，最后输出 10 个类别分数。不同任务共用隐藏层，各占输出层中相邻的两个位置。

这个网络只包含线性层和 ReLU，没有批归一化的运行统计量。后面的 EWC 因而只需处理权重和偏置。


In [ ]:
class SimpleMLP(nn.Module):
    """Two ReLU hidden layers without task-dependent running statistics."""
    def __init__(self, input_size=784, hidden_units=400, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_units)
        self.fc2 = nn.Linear(hidden_units, hidden_units)
        self.fc3 = nn.Linear(hidden_units, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


print(SimpleMLP(hidden_units=CONFIG['hidden_units']))

## 7. EWC：限制重要参数的变化

学习完一个任务后，EWC 保存当时的参数，并估计每个参数对该任务的重要性。这里使用经验 Fisher 对角线：对各训练样本计算交叉熵梯度，逐项平方后取平均。

后续训练中，参数偏离旧值越多、对应的重要性越高，产生的惩罚就越大。模型将各旧任务的惩罚相加，再乘上系数 λ，与当前任务的交叉熵一起优化。增加 λ 会加强约束，但也可能妨碍新任务学习。

重要性估计使用与训练相同的标签映射和输出范围，每个任务固定抽取至多 1024 个训练样本。日志分开显示交叉熵与加权 EWC 惩罚，便于观察两者的变化。


In [ ]:
class EWCModel(SimpleMLP):
    """Keep one empirical Fisher diagonal and parameter anchor per old task."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fisher_matrices = []
        self.prev_params = []

    def compute_fisher(self, dataloader, task_id, classes, scenario="task"):
        """Average squared per-example gradients using the training label map."""
        if dataloader.batch_size != 1:
            raise ValueError("Fisher estimation requires batch_size=1")
        self.eval()
        fisher = {name: torch.zeros_like(p) for name, p in self.named_parameters()}
        count = 0
        model_device = next(self.parameters()).device
        for x, y in dataloader:
            x, y = x.to(model_device), y.to(model_device)
            target = remap_labels(y, classes)
            loss = scenario_loss(self(x), target, task_id, scenario)
            grads = torch.autograd.grad(loss, self.parameters())
            for (name, _), grad in zip(self.named_parameters(), grads):
                fisher[name] += grad.detach().square()
            count += y.numel()
        return {name: value / count for name, value in fisher.items()}

    def ewc_loss(self):
        penalty = next(self.parameters()).new_zeros(())
        for fisher, anchor in zip(self.fisher_matrices, self.prev_params):
            for name, param in self.named_parameters():
                penalty = penalty + (fisher[name] * (param - anchor[name]).square()).sum()
        return penalty / 2

    def consolidate(self, fisher):
        """Retain this task's constraint alongside earlier tasks."""
        self.fisher_matrices.append(
            {name: value.detach().clone() for name, value in fisher.items()}
        )
        self.prev_params.append({name: p.detach().clone() for name, p in self.named_parameters()})


print("EWC 模型定义完成")

## 8. Task-IL 与 Class-IL 如何训练和评估

任务增量学习（Task-IL）在预测时已知任务编号，类别增量学习（Class-IL）则需要区分所有已见类别。这个区别也体现在训练损失中：

| 设定 | 训练与 Fisher 估计使用的输出 | 测试时的分类范围 |
| --- | --- | --- |
| Task-IL | 当前任务的两个输出 | 根据任务编号选择该任务的两个输出 |
| Class-IL | 截至当前阶段见过的所有类别 | 在所有已见类别中选择 |

网络提前分配十个输出，尚未见过的类别暂不参与损失或预测。任务内的两个标签先映射为 0、1；Class-IL 再加上 `2 * task_id`，得到它们在输出层中的位置。例如，第二个任务（`task_id=1`）对应输出位置 2、3，与原始数字标签无关。

两种设定分别从相同的初始参数开始训练，共享隐藏层始终参与更新。它们不仅是测试时选择输出的方式不同，也有各自的训练过程。


### 标签映射与分类损失

原始数字标签先映射到任务内的位置。Task-IL 的损失只使用当前任务的输出，Class-IL 的损失使用所有已见类别的输出。下面两个函数也用于 Fisher 估计，使标签含义保持一致。


In [ ]:
def remap_labels(y, task_classes):
    """Map digit labels to their positions in the task class list."""
    return torch.tensor(
        [task_classes.index(int(c.item())) for c in y], device=y.device, dtype=y.dtype
    )


def scenario_loss(output, local_labels, task_id, scenario):
    """Use the same active output range for training and Fisher estimation."""
    if scenario == "task":
        return F.cross_entropy(output[:, task_id * 2 : (task_id + 1) * 2], local_labels)
    if scenario == "class":
        return F.cross_entropy(output[:, : (task_id + 1) * 2], local_labels + task_id * 2)
    raise ValueError(f"Unknown scenario: {scenario}")


### 评估已学任务

评估时逐个读取已学任务的数据。`evaluate_scenario()` 统计一个批次的正确预测数，`evaluate_all_tasks()` 返回各任务准确率及按样本数加权的总体准确率。


In [ ]:
def evaluate_scenario(model, output, y_remapped, eval_task_id, task_id, scenario="task"):
    """Count correct predictions with a known task or among all seen classes."""
    if scenario == "task":
        pred = output[:, eval_task_id * 2 : (eval_task_id + 1) * 2].argmax(1)
        return (pred == y_remapped).sum().item()
    if scenario == "class":
        pred = output[:, : (task_id + 1) * 2].argmax(1)
        return (pred == y_remapped + eval_task_id * 2).sum().item()
    raise ValueError(f"Unknown scenario: {scenario}")


def evaluate_all_tasks(model, test_tasks, task_id, scenario="task"):
    """Return per-task and sample-weighted accuracy for the current scenario."""
    model.eval()
    accuracies, total_correct, total_count = [], 0, 0
    with torch.no_grad():
        for eval_task_id in range(task_id + 1):
            dataset, classes = test_tasks[eval_task_id]
            loader = DataLoader(dataset, batch_size=CONFIG["batch_size"], shuffle=False)
            correct, count = 0, 0
            for x, y in loader:
                x, y = x.to(CONFIG["device"]), y.to(CONFIG["device"])
                correct += evaluate_scenario(
                    model, model(x), remap_labels(y, classes), eval_task_id, task_id, scenario
                )
                count += y.numel()
            accuracies.append(correct / count)
            total_correct += correct
            total_count += count
    return accuracies, total_correct / total_count


### 保存与抽取旧样本

回放缓冲区保存在 CPU 上，用蓄水池抽样保留旧图像，并保存它们在 Class-IL 中的类别编号。每次训练从中均匀、无放回地抽取旧样本。


In [ ]:
class ReplayBuffer:
    """Keep a uniform reservoir on CPU and sample without replacement."""

    def __init__(self, capacity, seed):
        self.capacity = capacity
        self.generator = torch.Generator().manual_seed(seed)
        self.images = None
        self.labels = torch.empty(capacity, dtype=torch.long)
        self.seen = 0
        self.size = 0

    def add_task(self, dataset, classes, task_id):
        # Only the just-completed training task is available at this boundary.
        for image, digit in dataset:
            if self.images is None:
                self.images = torch.empty((self.capacity, *image.shape), dtype=image.dtype)
            self.seen += 1
            slot = (
                self.seen - 1
                if self.seen <= self.capacity
                else int(torch.randint(self.seen, (), generator=self.generator))
            )
            if slot < self.capacity:
                self.images[slot].copy_(image)
                self.labels[slot] = classes.index(int(digit)) + 2 * task_id
            self.size = min(self.seen, self.capacity)

    def sample(self, count):
        indices = torch.randperm(self.size, generator=self.generator)[:count]
        return self.images[indices], self.labels[indices]

    def storage_bytes(self):
        if self.images is None:
            return 0
        return (
            self.images.numel() * self.images.element_size()
            + self.labels.numel() * self.labels.element_size()
        )


### 训练当前任务

每个批次先准备图像和标签，再计算分类损失及 EWC 惩罚，最后更新参数。启用回放时，旧样本替换一部分当前样本；批量大小和更新次数保持不变。


In [ ]:
def train_task(
    model,
    train_loader,
    test_tasks,
    task_id,
    task_classes,
    ewc_lambda=1000,
    scenario="task",
    replay=None,
):
    """Keep update counts and batch sizes fixed when mixing in old examples."""
    if replay is not None and scenario != "class":
        raise ValueError("Replay comparison uses Class-IL")
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
    history = []
    for epoch in range(CONFIG["epochs_per_task"]):
        model.train()
        ce_sum, penalty_sum, count = 0.0, 0.0, 0
        updates, current_examples, replay_examples = 0, 0, 0
        for x, y in train_loader:
            replay_count = min(len(y) // 2, replay.size) if replay is not None else 0
            current_count = len(y) - replay_count
            if replay_count:
                old_x, old_y = replay.sample(replay_count)
                targets = torch.cat(
                    (remap_labels(y[:current_count], task_classes) + 2 * task_id, old_y)
                )
                x = torch.cat((x[:current_count], old_x))
            x, y = x.to(CONFIG["device"]), y.to(CONFIG["device"])
            output = model(x)
            if replay_count:
                ce_loss = F.cross_entropy(output[:, : 2 * (task_id + 1)], targets.to(output.device))
            else:
                ce_loss = scenario_loss(output, remap_labels(y, task_classes), task_id, scenario)
            penalty = (
                ewc_lambda * model.ewc_loss() if ewc_lambda and task_id else output.new_zeros(())
            )
            optimizer.zero_grad()
            (ce_loss + penalty).backward()
            optimizer.step()
            ce_sum += ce_loss.item() * y.numel()
            penalty_sum += penalty.item() * y.numel()
            count += y.numel()
            updates += 1
            current_examples += current_count
            replay_examples += replay_count
        scores, average = evaluate_all_tasks(model, test_tasks, task_id, scenario)
        row = dict(
            task=task_id,
            epoch=epoch,
            cross_entropy=ce_sum / count,
            weighted_ewc=penalty_sum / count,
            current_accuracy=scores[-1],
            seen_accuracy=average,
            updates=updates,
            current_examples=current_examples,
            replay_examples=replay_examples,
        )
        history.append(row)
        print(
            f"  {scenario}-IL epoch {epoch + 1}: CE={row['cross_entropy']:.4f}, "
            f"weighted EWC={row['weighted_ewc']:.4f}, "
            f"new={scores[-1]:.2%}, seen={average:.2%}"
        )
    return history


## 9. 固定更新次数，比较 EWC 与回放

下面共运行七次实验：Task-IL 与 Class-IL 各使用 λ=0、100、1000，另外增加一组 Class-IL 样本回放（Replay）。λ=0 对应普通顺序训练，Replay 也不加 EWC 惩罚。

所有实验复制相同的初始网络，沿用相同的任务顺序、学习率、批量大小和每任务更新次数。λ 与回放容量在训练前指定，测试集用于评估这些设置的结果。

Replay 在 CPU 上保存至多 500 张旧训练图像及其类别标签。每个后续任务开始前，缓冲区通过蓄水池抽样纳入刚学完的任务：每个已见训练样本的保留概率相同，各类别的数量可能不同。

从第二个任务起，每个打乱后的批次中，约一半当前样本被旧样本替换。旧样本均匀、无放回抽取；缓冲区不足时减少替换量。这样保持了批量大小和更新次数，但当前任务样本参与训练的次数会减少。回放使用独立随机数生成器，不改变当前数据的打乱顺序。

每学完一个任务，就评估所有已学任务。下方同时记录新旧样本使用次数和缓冲区存储量。EWC 还有估计 Fisher 的额外计算，因此这里固定的是梯度更新次数，并非总计算开销。


In [ ]:
num_classes = CONFIG["num_tasks"] * CONFIG["classes_per_task"]
seed_everything(SEED, deterministic=True)
initial_model = EWCModel(hidden_units=CONFIG["hidden_units"], num_classes=num_classes)
initial_state = {name: value.clone() for name, value in initial_model.state_dict().items()}
initial_digest = parameter_digest(initial_model)
EWC_WEIGHTS = (0.0, 100.0, CONFIG["ewc_lambda"])
results = {"task": {}, "class": {}}

for scenario in ("task", "class"):
    methods = list(EWC_WEIGHTS) + (["replay"] if scenario == "class" else [])
    for method in methods:
        penalty_weight = 0.0 if method == "replay" else method
        replay = ReplayBuffer(CONFIG["replay_capacity"], SEED + 200) if method == "replay" else None
        seed_everything(SEED, deterministic=True)
        model = EWCModel(hidden_units=CONFIG["hidden_units"], num_classes=num_classes).to(device)
        model.load_state_dict(initial_state)
        assert parameter_digest(model) == initial_digest
        result = dict(
            initial_digest=initial_digest,
            scenario=scenario,
            ewc_lambda=penalty_weight,
            method="Replay" if replay is not None else f"EWC lambda={penalty_weight:g}",
            matrix=[],
            average=[],
            history=[],
            memory=[],
        )
        print(f"\n{scenario}-IL {result['method']}, initialization={initial_digest[:12]}")
        for task_id in range(CONFIG["num_tasks"]):
            train_dataset, classes = train_tasks[task_id]
            train_loader = DataLoader(
                train_dataset,
                batch_size=CONFIG["batch_size"],
                shuffle=True,
                generator=torch.Generator().manual_seed(SEED + task_id),
            )
            result["history"].extend(
                train_task(
                    model,
                    train_loader,
                    test_tasks,
                    task_id,
                    classes,
                    ewc_lambda=penalty_weight,
                    scenario=scenario,
                    replay=replay,
                )
            )
            scores, average = evaluate_all_tasks(model, test_tasks, task_id, scenario)
            result["matrix"].append(scores)
            result["average"].append(average)
            if penalty_weight and task_id < CONFIG["num_tasks"] - 1:
                fisher_indices = torch.randperm(
                    len(train_dataset),
                    generator=torch.Generator().manual_seed(SEED + 100 + task_id),
                )[: CONFIG["fisher_samples"]].tolist()
                fisher_loader = DataLoader(
                    Subset(train_dataset, fisher_indices),
                    batch_size=CONFIG["fisher_batch_size"],
                    shuffle=False,
                )
                model.consolidate(model.compute_fisher(fisher_loader, task_id, classes, scenario))
            if replay is not None and task_id < CONFIG["num_tasks"] - 1:
                replay.add_task(train_dataset, classes, task_id)
            result["memory"].append(
                dict(
                    images=replay.size if replay is not None else 0,
                    bytes=replay.storage_bytes() if replay is not None else 0,
                )
            )
        results[scenario][method] = result

accuracy_matrix_task_il = results["task"][CONFIG["ewc_lambda"]]["matrix"]
accuracy_matrix_class_il = results["class"][CONFIG["ewc_lambda"]]["matrix"]
avg_accs_task_il = results["task"][CONFIG["ewc_lambda"]]["average"]
avg_accs_class_il = results["class"][CONFIG["ewc_lambda"]]["average"]
print("下方热图展示两套独立协议在 lambda=1000 时的结果；汇总表包含全部七次实验和 Replay 矩阵。")


## 10. 读取准确率矩阵

第 $i$ 行表示学完第 $i$ 个任务后的状态，第 $j$ 列表示在第 $j$ 个任务测试集上的准确率。对角线记录任务刚学完时的表现；沿同一列向下看，可以观察它在后续训练中的变化。上三角对应尚未评估的任务，图中留空。

如果某列的对角线数值已经很低，之后保持低值不能说明模型成功保留了该任务。应先看任务是否学会，再看准确率下降了多少。


In [ ]:
print("\n" + "=" * 80)
print(" 准确率矩阵")
print("=" * 80)

for scenario, accuracy_matrix, avg_accs in [
    ("TASK-IL", accuracy_matrix_task_il, avg_accs_task_il),
    ("CLASS-IL", accuracy_matrix_class_il, avg_accs_class_il),
    (
        "CLASS-IL Replay",
        results["class"]["replay"]["matrix"],
        results["class"]["replay"]["average"],
    ),
]:
    print(f"\n{scenario}:")
    print(f"{'任务':<8}", end="")
    for i in range(CONFIG["num_tasks"]):
        print(f"T{i + 1:<10}", end="")
    print(f"{'Avg':<10}")
    print("-" * (8 + 10 * CONFIG["num_tasks"] + 10))

    for i, row in enumerate(accuracy_matrix):
        print(f"T{i + 1:<8}", end="")
        for j, acc in enumerate(row):
            print(f"{acc:<10.4f}", end="")
        remaining_cols = CONFIG["num_tasks"] - len(row)
        print(" " * (10 * remaining_cols), end="")
        print(f"{avg_accs[i]:<10.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (scenario, accuracy_matrix) in enumerate(
    [
        ("TASK-IL", accuracy_matrix_task_il),
        ("CLASS-IL", accuracy_matrix_class_il),
        ("CLASS-IL Replay", results["class"]["replay"]["matrix"]),
    ]
):
    # Row i stores the scores after training task i.
    matrix = np.zeros((CONFIG["num_tasks"], CONFIG["num_tasks"]))
    for i, row in enumerate(accuracy_matrix):
        for j, acc in enumerate(row):
            matrix[i, j] = acc

    # Hide tasks that have not been evaluated at this stage.
    mask = np.triu(np.ones_like(matrix, dtype=bool), k=1)

    image = axes[idx].imshow(np.ma.array(matrix, mask=mask), cmap="YlGn", vmin=0, vmax=1)
    fig.colorbar(image, ax=axes[idx], label="Accuracy")
    for row in range(CONFIG["num_tasks"]):
        for col in range(row + 1):
            axes[idx].text(col, row, f"{matrix[row, col]:.3f}", ha="center", va="center")
    axes[idx].set_title(f"{scenario} - Accuracy Matrix", fontsize=14, fontweight="bold")
    axes[idx].set_xlabel("Evaluation Task")
    axes[idx].set_ylabel("Training Task")
    axes[idx].set_xticks(
        range(CONFIG["num_tasks"]), [f"T{i + 1}" for i in range(CONFIG["num_tasks"])]
    )
    axes[idx].set_yticks(
        range(CONFIG["num_tasks"]), [f"T{i + 1}" for i in range(CONFIG["num_tasks"])]
    )

plt.tight_layout()
plt.show()


## 11. 汇总准确率与遗忘量

本页使用以下定义：

- **最终总体准确率**：最后一个任务学完后，在全部任务测试样本上统计的准确率。它按各任务的样本数加权。
- **任务遗忘量**：刚学完该任务时的准确率减去最终准确率。正值表示下降，负值表示后续有所提高。
- **向后转移（Backward Transfer，BWT）**：最终准确率减去刚学完时的准确率。

遗忘量和 BWT 都对前四个任务取平均。在本页的定义下，两者互为相反数；最后一个任务没有后续训练阶段，因此不计入。


In [ ]:
def compute_metrics(accuracy_matrix, avg_acc_per_task, num_tasks):
    """Summarize final accuracy, forgetting, and backward transfer."""
    metrics = {}

    # Each stage average is weighted by the number of test examples.
    metrics["avg_acc_per_task"] = avg_acc_per_task

    # Use the pooled test accuracy after the final task.
    metrics["avg_final_acc"] = avg_acc_per_task[-1]

    # Forgetting = accuracy just after learning - final accuracy.
    forgetting = []
    for i in range(num_tasks - 1):
        learned_acc = accuracy_matrix[i][i]
        final_acc = accuracy_matrix[-1][i]
        forgetting.append(learned_acc - final_acc)
    metrics["avg_forgetting"] = np.mean(forgetting) if forgetting else 0
    metrics["forgetting_per_task"] = forgetting

    # BWT reverses that difference; both metrics exclude the final task.
    bwt = []
    for i in range(num_tasks - 1):
        bwt.append(accuracy_matrix[-1][i] - accuracy_matrix[i][i])
    metrics["avg_bwt"] = np.mean(bwt) if bwt else 0
    metrics["bwt_per_task"] = bwt

    return metrics


metrics_task_il = compute_metrics(accuracy_matrix_task_il, avg_accs_task_il, CONFIG["num_tasks"])
metrics_class_il = compute_metrics(accuracy_matrix_class_il, avg_accs_class_il, CONFIG["num_tasks"])

print("\n" + "=" * 80)
print("持续学习指标")
print("=" * 80)

for scenario, metrics in [("TASK-IL", metrics_task_il), ("CLASS-IL", metrics_class_il)]:
    print(f"\n{scenario}:")
    print(f"  平均最终准确率: {metrics['avg_final_acc']:.4f}")
    print(f"  平均遗忘 (Forgetting): {metrics['avg_forgetting']:.4f}")
    print(f"  平均向后转移 (BWT): {metrics['avg_bwt']:.4f}")


print("\nMatched comparison")
for scenario, weights in results.items():
    for weight, result in weights.items():
        metrics = compute_metrics(result["matrix"], result["average"], CONFIG["num_tasks"])
        metrics["plasticity"] = float(np.mean([row[i] for i, row in enumerate(result["matrix"])]))
        metrics["old_task_retention"] = float(np.mean(result["matrix"][-1][:-1]))
        result["metrics"] = metrics
        print(
            f"{scenario}-IL {result['method']}: final={metrics['avg_final_acc']:.2%}, "
            f"new-at-learning={metrics['plasticity']:.2%}, "
            f"old-at-end={metrics['old_task_retention']:.2%}, "
            f"forgetting={metrics['avg_forgetting']:.2%}"
        )

print("\nTraining budget and replay storage")
for scenario, methods in results.items():
    for result in methods.values():
        history = result["history"]
        print(
            f"{scenario}-IL {result['method']}: "
            f"updates={sum(row['updates'] for row in history)}, "
            f"current={sum(row['current_examples'] for row in history)}, "
            f"replayed={sum(row['replay_examples'] for row in history)}, "
            f"stored images={max(row['images'] for row in result['memory'])}, "
            f"buffer bytes={max(row['bytes'] for row in result['memory'])}"
        )


## 12. 观察新任务学习与旧任务保持

第一幅图分别展示 Task-IL 和 Class-IL 中改变 λ 的结果。**可塑性**用各任务刚学完时的准确率均值表示，**旧任务保持**用最后阶段前四个任务的准确率均值表示；二者都按任务等权平均，总体准确率则按样本数加权。

先看各任务刚学完的准确率，再看最后还保留多少。如果增大 λ 后旧任务下降较少，但新任务也学不好，就说明约束同时限制了学习。

第二幅图比较 Class-IL 下的普通顺序训练、EWC（λ=1000）和 Replay。这里关注回放能改善多少旧任务准确率，以及这种改善是否伴随新任务准确率下降。容量 500 是本次选定的设置，要了解结果是否稳定，还需要改变训练种子重新实验。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (scenario, weights) in zip(axes, results.items()):
    for metric, label in (("plasticity", "New task at learning"),
                          ("old_task_retention", "Old tasks at end"),
                          ("avg_final_acc", "All tasks at end")):
        ax.plot(range(len(EWC_WEIGHTS)), [weights[w]['metrics'][metric] for w in EWC_WEIGHTS],
                marker="o", label=label)
    ax.set_xticks(range(len(EWC_WEIGHTS)), [f"{w:g}" for w in EWC_WEIGHTS])
    ax.set(xlabel="EWC lambda", ylabel="Accuracy", ylim=(0, 1), title=f"{scenario}-IL")
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
comparison_methods = (0.0, CONFIG['ewc_lambda'], "replay")
positions = np.arange(len(comparison_methods))
for offset, (metric, label) in enumerate((("plasticity", "New task at learning"),
                                         ("old_task_retention", "Old tasks at end"),
                                         ("avg_final_acc", "All tasks at end"))):
    ax.bar(positions + (offset - 1) * 0.25,
           [results['class'][method]['metrics'][metric] for method in comparison_methods],
           width=0.25, label=label)
ax.set_xticks(positions, ["Sequential", f"EWC ({CONFIG['ewc_lambda']:g})", "Replay"])
ax.set(ylabel="Accuracy", ylim=(0, 1), title="Class-IL: fixed update and batch budgets")
ax.legend()
plt.tight_layout()
plt.show()


## 可选扩展

- 可以预先选定另一种回放容量，保持批量大小和更新次数一致，观察多保存一些旧样本能改善多少旧任务准确率，以及新任务学习是否受到影响。
- 可以改变 EWC 的强度，在 Task-IL 和 Class-IL 中分别观察新任务学习与旧任务保持的变化。测试集用于比较预先选定的设置，不用于挑选最好的参数。
